# Colab A: Evaluation
Standalone. Sequential extract-eval-cleanup to fit 70GB free-tier disk.
Run in parallel with Colab B (ablation).

# GLADA: Multi-Dataset Evaluation

**Grouped Layer Adversarial Domain Adaptation for Audio Deepfake Detection**

Jakob Kauffmann | MS CS Thesis | San José State University | Advisor: Dr. Faranak Abri

---

## Purpose

This notebook evaluates trained GLADA checkpoints across all standard audio deepfake detection benchmarks. While `glada_experiment.ipynb` handles training and primary evaluation on In-the-Wild (ITW), this notebook provides comprehensive cross-dataset evaluation needed for:

1. **SOTA comparison tables** — comparing GLADA against published results (IB-CAAN, HierCon, AASIST, etc.)
2. **Generalization analysis** — measuring how domain adaptation improves performance across different types of distribution shift
3. **Multi-model comparison** — comparing different backbones, seeds, and ablation variants

### Evaluation Datasets

| Dataset | Type | Shift Type | Key Reference |
|---------|------|-----------|---------------|
| In-the-Wild (ITW) | Real-world deepfakes | Full domain shift | Müller et al. (2024) |
| ASVspoof 2019 LA Eval | Lab conditions | In-domain | Wang et al. (2020) |
| ASVspoof 2021 LA Eval | Lab + codecs | Codec compression | Yamagishi et al. (2021) |
| ASVspoof 2021 DF Eval | Deepfake track | Novel attacks | Yamagishi et al. (2021) |
| WaveFake | Neural vocoders | Vocoder diversity | Frank & Schönherr (2021) |

### Prerequisites
- Trained GLADA checkpoint(s) from `glada_experiment.ipynb`
- Evaluation datasets extracted to `/content/data/`

## 0. Environment Setup

Mount Google Drive (for checkpoint access) and install required packages. This cell must be run first in every new Colab session.

**Note on `num_workers=0`**: Colab's multiprocessing environment does not support `fork`-based worker processes for DataLoaders when CUDA is initialized. All DataLoaders in this notebook use `num_workers=0` (single-process loading) to avoid `RuntimeError: DataLoader worker exited unexpectedly`.

In [25]:
# ============================================================================
# 0. Environment Setup
# Install dependencies, mount Google Drive, and verify GPU availability.
# This cell is self-contained so the notebook works in a fresh Colab session.
# ============================================================================

# Install required packages (quiet mode to reduce output noise)
!pip install -q transformers soundfile librosa scikit-learn tqdm seaborn

# Mount Google Drive for checkpoint and dataset access
from google.colab import drive
drive.mount('/content/drive')

# GPU diagnostics — evaluation is feasible on T4 but faster on A100
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name()
    capability = torch.cuda.get_device_capability()
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'Compute capability: {capability}')
    print(f'Memory: {mem_gb:.1f} GB')
    # Enable TF32 on Ampere+ GPUs (A100, etc.) for faster matmul
    if capability[0] >= 8:
        torch.set_float32_matmul_precision('high')
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cuda.matmul.allow_tf32 = True
        print('TF32 enabled for Ampere+ GPU')
else:
    print('WARNING: No GPU detected. Evaluation will be very slow.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB
Compute capability: (8, 0)
Memory: 42.4 GB
TF32 enabled for Ampere+ GPU


## Imports and Configuration

All imports needed for evaluation. The model architecture classes must exactly match those used during training to ensure checkpoint compatibility — `torch.load()` reconstructs the state dict using the same class definitions, so any mismatch in layer names, dimensions, or module structure will cause loading failures.

We also define the `BACKBONE_REGISTRY` and `GLADAConfig` dataclass here, identically to `glada_experiment.ipynb`, so that checkpoints trained with any backbone can be loaded and evaluated without importing external `.py` files (keeping the notebook fully self-contained per project convention).

**Key dependencies**: HuggingFace `transformers` (Wolf et al., 2020) for SSL backbone loading, `soundfile` for audio I/O, `librosa` (McFee et al., 2015) for resampling, and `scikit-learn` (Pedregosa et al., 2011) for EER/AUC computation.

In [26]:
# ============================================================================
# 1. Imports + Backbone Registry + GLADAConfig
# These must exactly match glada_experiment.ipynb for checkpoint compatibility.
# ============================================================================
import os, random, json, time, warnings, logging, math, copy
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Tuple, List, Dict

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

# Audio loading: always soundfile, never torchaudio
try:
    import soundfile as sf
    HAS_SOUNDFILE = True
except ImportError:
    HAS_SOUNDFILE = False
    print('WARNING: soundfile not available, falling back to torchaudio')

import librosa
from transformers import AutoModel, AutoConfig
from sklearn.metrics import roc_curve, roc_auc_score
from tqdm.auto import tqdm

# Visualization imports for result analysis
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')

# ============================================================================
# Backbone Registry
# Maps backbone names to HuggingFace model IDs and architecture params.
# num_groups determines how SSL layers are partitioned for GLADA.
# ============================================================================
BACKBONE_REGISTRY = {
    'wavlm-large': {
        'hf_id': 'microsoft/wavlm-large',
        'dim': 1024, 'num_layers': 24, 'num_groups': 4,
    },
    'xlsr-300m': {
        'hf_id': 'facebook/wav2vec2-xls-r-300m',
        'dim': 1024, 'num_layers': 24, 'num_groups': 4,
    },
    'data2vec-large': {
        'hf_id': 'facebook/data2vec-audio-large-960h',
        'dim': 1024, 'num_layers': 24, 'num_groups': 4,
    },
    'wav2vec2-robust': {
        'hf_id': 'facebook/wav2vec2-large-robust',
        'dim': 1024, 'num_layers': 24, 'num_groups': 4,
    },
    'wavlm-base-plus': {
        'hf_id': 'microsoft/wavlm-base-plus',
        'dim': 768, 'num_layers': 12, 'num_groups': 3,
    },
    'xlsr-1b': {
        'hf_id': 'facebook/wav2vec2-xls-r-1b',
        'dim': 1280, 'num_layers': 48, 'num_groups': 6,
    },
}

# ============================================================================
# GLADAConfig — Central configuration with ablation switches
# Must be identical to glada_experiment.ipynb for checkpoint compatibility.
# ============================================================================

@dataclass
class GLADAConfig:
    # --- Experiment ---
    experiment_name: str = 'glada'
    run_id: str = field(default_factory=lambda: datetime.now().strftime('%Y%m%d_%H%M%S'))
    seed: int = 42

    # --- Backbone ---
    backbone: str = 'wavlm-large'  # key into BACKBONE_REGISTRY

    # --- Architecture ---
    bottleneck_dim: int = 256
    vib_beta: float = 1e-4          # KL weight (Alemi et al., ICLR 2017)
    num_speakers: int = 107         # ASVspoof 2019 LA train speakers

    # --- Loss weights ---
    domain_loss_weight: float = 0.1  # alpha (>0.1 destabilizes, tested)
    speaker_loss_weight: float = 0.1 # gamma (max, scheduled)
    cka_loss_weight: float = 0.01    # delta (Kornblith et al., ICML 2019)

    # --- GRL scheduling (Ganin et al., JMLR 2016) ---
    grl_warmup_epochs: int = 5       # domain GRL warmup
    speaker_grl_warmup_epochs: int = 10  # speaker GRL warmup (slower)

    # --- Ablation switches ---
    use_vib: bool = True              # Toggle VIB (deterministic projection if False)
    use_speaker_adversary: bool = True  # Toggle speaker GRL
    use_global_domain_disc: bool = False  # Single domain disc on z (baseline comparison)
    use_cka_loss: bool = True         # Toggle CKA diversity between groups
    use_hierarchical: bool = True     # Grouped attention (flat weighted sum if False)
    use_rawboost: bool = True         # Toggle RawBoost augmentation (Tak et al., 2022)
    use_glada: bool = True            # Per-group domain discriminators (NOVEL)

    # --- GLADA architecture ---
    glada_disc_hidden: int = 128      # Hidden dim for per-group domain discriminators

    # --- Training ---
    phase1_epochs: int = 5
    phase2_epochs: int = 45
    batch_size: int = 24             # per domain
    effective_batch_size: int = 384
    grad_accum_steps: int = 1
    head_lr: float = 1e-3
    backbone_lr: float = 1e-5       # phase 2 peak
    llrd_decay: float = 0.95         # Layer-wise LR decay (Clark et al., EMNLP 2020)
    weight_decay: float = 0.01
    gradient_clip: float = 1.0
    patience: int = 15
    use_amp: bool = True
    amp_dtype: str = 'bfloat16'     # bfloat16 for A100, float16 for T4

    # --- Data ---
    max_audio_len: int = 64000      # 4s at 16kHz
    rawboost_prob: float = 0.5
    num_workers: int = 0

    # --- Paths ---
    drive_log_dir: str = '/content/drive/MyDrive/Jakob/Experiments'

    def __post_init__(self):
        bb = BACKBONE_REGISTRY[self.backbone]
        self.hf_model_id = bb['hf_id']
        self.hidden_dim = bb['dim']
        self.num_layers = bb['num_layers']
        self.num_groups = bb['num_groups']
        self.total_hidden_states = self.num_layers + 1  # +1 for CNN embedding layer
        actual_batch = self.batch_size * 2  # source + target
        self.grad_accum_steps = max(1, self.effective_batch_size // actual_batch)
        self.total_epochs = self.phase1_epochs + self.phase2_epochs

    def get_run_dir(self):
        return os.path.join(self.drive_log_dir, self.experiment_name, self.run_id)


def set_seed(seed):
    """Reproducibility across all RNGs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


print('Imports complete. Config and backbone registry loaded.')

PyTorch 2.10.0+cu128
GPU: NVIDIA A100-SXM4-40GB
Imports complete. Config and backbone registry loaded.


## Dataset Paths and Extraction

Colab ephemeral storage is wiped between sessions. This cell verifies dataset availability and extracts from Google Drive archives if needed. All datasets are stored as compressed archives in Drive for persistence.

**Note**: ASVspoof 2021 LA and DF evaluation sets (Yamagishi et al., 2021) use a different protocol format than 2019 LA (Wang et al., 2020) — they provide only audio keys without ground truth labels during the challenge, but ground truth keys files are now publicly available.

### Dataset Sources

| Dataset | Archive Format | Approx. Size | Audio Format |
|---------|---------------|-------------|-------------|
| ASVspoof 2019 LA (Wang et al., 2020) | `.tar` | ~5 GB | FLAC, 16kHz |
| ASVspoof 2021 LA Eval (Yamagishi et al., 2021) | `.tar.gz` | ~2 GB | FLAC, various codecs |
| ASVspoof 2021 DF Eval (Yamagishi et al., 2021) | `.tar.gz` | ~3 GB | FLAC, deepfakes |
| In-the-Wild (M\u00fcller et al., 2024) | `.tar` | ~1.5 GB | Various formats |
| WaveFake (Frank & Sch\u00f6nherr, 2021) | `.tar.gz` | ~4 GB | WAV, 22050Hz (resampled to 16kHz) |

The extraction function handles both `.tar` and `.tar.gz` formats, skipping extraction if data is already present (idempotent). This is critical for Colab sessions where re-running cells should not re-extract gigabytes of data.

In [27]:
# ============================================================================
# 2. Dataset Paths and Sequential Extract-Evaluate-Cleanup
# To fit within 70GB free-tier disk, we extract one dataset at a time,
# evaluate it, then delete the local copy before extracting the next.
# ============================================================================
import tarfile
import shutil

LOCAL_ROOT = '/content/data'
os.makedirs(LOCAL_ROOT, exist_ok=True)

DRIVE_BASE = '/content/drive/MyDrive/Jakob/Datasets'
CODECFAKE_BASE = '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus'

EVAL_BATCH_SIZE = 48  # Larger batch for eval (no gradients stored)

def extract_dataset(tar_path, extract_to, name):
    """Extract a tar/tar.gz archive to local storage."""
    if not os.path.exists(tar_path):
        print(f'  [{name}] Archive not found: {tar_path} — SKIPPING')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted at {extract_to}')
        return True
    print(f'  [{name}] Extracting from {tar_path}...')
    mode = 'r:gz' if tar_path.endswith('.gz') else 'r:*'
    with tarfile.open(tar_path, mode) as tar:
        tar.extractall(path=extract_to)
    print(f'  [{name}] Done.')
    return True

def cleanup_dataset(local_dir, name):
    """Remove extracted dataset to free disk space."""
    if os.path.exists(local_dir):
        shutil.rmtree(local_dir)
        print(f'  [{name}] Cleaned up {local_dir}')

def extract_multipart_dataset(part_pattern, extract_to, name):
    """Concatenate multipart tar.gz archives and extract.

    ASVspoof 2021 DF eval is split into parts on Drive due to size:
      ASVspoof2021_DF_eval_part_00, ASVspoof2021_DF_eval_part_01, ...
    These must be concatenated into a single tar.gz before extraction.
    """
    import glob
    parts = sorted(glob.glob(part_pattern))
    if not parts:
        print(f'  [{name}] No parts found matching {part_pattern} — SKIPPING')
        return False
    os.makedirs(extract_to, exist_ok=True)
    if os.listdir(extract_to):
        print(f'  [{name}] Already extracted at {extract_to}')
        return True

    concat_path = os.path.join(LOCAL_ROOT, f'{name}_concat.tar.gz')
    print(f'  [{name}] Concatenating {len(parts)} parts...')
    with open(concat_path, 'wb') as out_f:
        for p in parts:
            print(f'    {os.path.basename(p)}')
            with open(p, 'rb') as in_f:
                shutil.copyfileobj(in_f, out_f)

    print(f'  [{name}] Extracting concatenated archive...')
    with tarfile.open(concat_path, 'r:gz') as tar:
        tar.extractall(path=extract_to)

    # Remove concat file to free disk
    os.remove(concat_path)
    print(f'  [{name}] Done. Removed temp concat file.')
    return True

def find_keys_file(base_dir, track='LA'):
    """Auto-discover the keys/protocol file for ASVspoof 2021.

    The extraction structure varies by archive version. Common locations:
      - keys/LA/CM/trial_metadata.txt
      - ASVspoof2021_LA_eval/keys/LA/CM/trial_metadata.txt
      - LA/keys/CM/trial_metadata.txt
    """
    import glob
    patterns = [
        f'{base_dir}/**/trial_metadata.txt',
        f'{base_dir}/**/keys/{track}/CM/trial_metadata.txt',
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

def find_audio_dir(base_dir, track='LA'):
    """Auto-discover the audio directory for ASVspoof 2021."""
    import glob
    patterns = [
        f'{base_dir}/**/ASVspoof2021_{track}_eval/flac',
        f'{base_dir}/**/flac',
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return None

# Dataset registry: each entry defines how to extract, create dataloader, and cleanup.
# 'no_cleanup' prevents deletion — used when the next dataset shares the same archive.
DATASET_REGISTRY = [
    {
        'name': 'ITW',
        'archive': os.path.join(DRIVE_BASE, 'release_in_the_wild.tar'),
        'local_dir': os.path.join(LOCAL_ROOT, 'itw'),
        'make_loader': lambda: DataLoader(
            InTheWildDataset(
                '/content/data/itw/release_in_the_wild/meta.csv',
                '/content/data/itw/release_in_the_wild',
                max_len=config.max_audio_len, use_labels=True),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2019_LA_eval',
        'archive': os.path.join(DRIVE_BASE, 'LA.tar'),
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof'),
        'no_cleanup': True,  # keep for 2019_LA_dev next
        'make_loader': lambda: DataLoader(
            ASVspoofEvalDataset(
                '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
                '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2019_LA_dev',
        'archive': None,  # reuses asvspoof extraction from above
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof'),  # cleanup this dir after dev
        'make_loader': lambda: DataLoader(
            ASVspoofEvalDataset(
                '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
                '/content/data/asvspoof/LA/ASVspoof2019_LA_dev/flac',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
    {
        'name': '2021_LA',
        'archive': os.path.join(DRIVE_BASE, 'ASVspoof2021_LA_eval.tar.gz'),
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_la'),
        'make_loader': 'auto_2021_LA',  # resolved at eval time via find_keys_file
    },
    {
        'name': '2021_DF',
        'archive': 'multipart',  # special: concatenate parts then extract
        'local_dir': os.path.join(LOCAL_ROOT, 'asvspoof21_df'),
        'make_loader': 'auto_2021_DF',  # resolved at eval time via find_keys_file
    },
    {
        'name': 'WaveFake',
        'archive': os.path.join(DRIVE_BASE, 'WaveFake.tar.gz'),
        'local_dir': os.path.join(LOCAL_ROOT, 'wavefake'),
        'make_loader': lambda: DataLoader(
            WaveFakeDataset(
                '/content/data/wavefake',
                max_len=config.max_audio_len),
            batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True),
    },
]

print(f"Registered {len(DATASET_REGISTRY)} datasets for sequential evaluation.")
print("Datasets will be extracted, evaluated, and cleaned up one at a time.")


Registered 6 datasets for sequential evaluation.
Datasets will be extracted, evaluated, and cleaned up one at a time.


## Audio Loading

Audio is loaded at 16kHz mono using `soundfile` (not `torchaudio` because `torchaudio.set_audio_backend()` is deprecated/removed in recent versions, and `torchaudio.load()` has had inconsistent behavior across Colab environments). All utterances are padded or truncated to 4 seconds (64,000 samples) to match training conditions.

The 16kHz sample rate matches the pretraining rate of all SSL backbones used in GLADA: WavLM (Chen et al., 2022), XLS-R (Babu et al., 2022), Data2Vec (Baevski et al., 2022), and Wav2Vec 2.0 (Baevski et al., 2020). Input length of 4 seconds follows the convention established by AASIST (Jung et al., 2022) and adopted by subsequent work including IB-CAAN (Huang et al., 2025).

**Key design decisions:**
- **Fixed length (4s)**: Matches training configuration. Longer utterances are truncated from a random start position during training, but from the beginning during evaluation for determinism.
- **Stereo to mono**: Some ITW and WaveFake files are stereo; we average channels.
- **Resampling**: WaveFake (Frank & Sch\u00f6nherr, 2021) audio is 22,050 Hz (LJSpeech native rate); we resample to 16,000 Hz using `librosa.resample()` (McFee et al., 2015) to match the SSL backbone's expected input rate.
- **Zero-padding**: Short utterances are right-padded with zeros; the attention mask tracks which positions are real audio vs. padding.

In [28]:
# ============================================================================
# 3. Audio Loading
# Deterministic loading for evaluation: no random cropping, always start at 0.
# Uses soundfile (not torchaudio) per project convention.
# ============================================================================

def load_audio(path, max_len=64000):
    """Load audio file, resample to 16kHz mono, pad/truncate to max_len.

    Args:
        path: Path to audio file (FLAC, WAV, or other soundfile-supported format)
        max_len: Target length in samples (default: 64000 = 4s at 16kHz)

    Returns:
        Tensor of shape (max_len,) with float32 audio samples
    """
    try:
        if HAS_SOUNDFILE:
            wav, sr = sf.read(str(path), dtype='float32')
            # Handle stereo -> mono (average channels)
            if len(wav.shape) > 1:
                wav = wav.mean(axis=1)
            # Resample if not 16kHz (e.g., WaveFake at 22050Hz)
            if sr != 16000:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
            wav = torch.from_numpy(wav)
        else:
            # Fallback to torchaudio (not preferred)
            import torchaudio
            wav, sr = torchaudio.load(str(path))
            if sr != 16000:
                wav = torchaudio.transforms.Resample(sr, 16000)(wav)
            wav = wav.mean(0) if wav.shape[0] > 1 else wav.squeeze(0)
    except Exception as e:
        # Return silence on load failure (logged but non-fatal)
        print(f'  WARNING: Failed to load {path}: {e}')
        wav = torch.zeros(max_len)

    # For evaluation: deterministic truncation from start (no random crop)
    if wav.shape[0] > max_len:
        wav = wav[:max_len]
    else:
        # Right-pad with zeros to fixed length
        wav = F.pad(wav, (0, max(0, max_len - wav.shape[0])))

    return wav


# Quick test: verify soundfile is available
print(f'soundfile available: {HAS_SOUNDFILE}')
print(f'Target audio length: 64000 samples (4.0s at 16kHz)')

soundfile available: True
Target audio length: 64000 samples (4.0s at 16kHz)


## Dataset Classes

> **Rule 13 Note**: The canonical dataset implementations live in `Coding/glada/datasets/` and are imported in new-style notebooks as `from glada.datasets.asvspoof import ASVspoofEvalDataset`. The classes below are **kept in this notebook for standalone portability** — this notebook can be run independently of the `glada` package installation by defining the classes locally. If the package is installed, prefer importing from `glada.datasets` directly.

Each evaluation dataset has its own class handling its specific protocol/metadata format. All return the same dict structure for unified evaluation:
- `input_values`: Tensor of shape `(max_len,)` — raw waveform at 16kHz
- `attention_mask`: Tensor of shape `(max_len,)` — 1 for real audio samples, 0 for zero-padding
- `task_label`: Integer tensor — 1 for bonafide, 0 for spoof

**Audio loading**: All classes use `soundfile.read()` (not `torchaudio.load()`).

This unified interface allows `MultiDatasetEvaluator` to iterate over any dataset without dataset-specific branching.

### Supported Datasets

| Class | Protocol | Notes |
|---|---|---|
| `ASVspoofEvalDataset` | ASVspoof 2019 LA eval | Wang et al., ICASSP 2020 |
| `ASVspoof2021Dataset` | ASVspoof 2021 LA or DF eval | Yamagishi et al., 2021 |
| `InTheWildDataset` | ITW via meta.csv | Muller et al., Interspeech 2022 |
| `WaveFakeDataset` | WaveFake (6 vocoders + LJSpeech) | Frank & Schonherr, NeurIPS 2021 |
| `CodecFakeDataset` | CodecFake+ | Codec-based attacks, 2024 |

In [29]:
# ============================================================================
# 4. Dataset Classes for All Evaluation Benchmarks
# Each class handles its own protocol format but returns a unified dict.
# ============================================================================


class ASVspoofEvalDataset(Dataset):
    """ASVspoof 2019 LA evaluation set (Wang et al., 2020).

    Protocol format: speaker_id audio_id - attack_type label
    Labels: 'bonafide' -> 1, 'spoof' -> 0
    Audio: FLAC files at 16kHz in the eval/flac directory.
    """

    def __init__(self, protocol_path, audio_dir, max_len=64000, max_samples=None):
        self.audio_dir = Path(audio_dir)
        self.max_len = max_len
        self.samples = []

        with open(protocol_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) >= 5:
                    self.samples.append({
                        'filename': p[1],
                        'label': 1 if p[4] == 'bonafide' else 0,
                        'attack': p[3],  # attack type (A07-A19 or '-' for bonafide)
                    })

        if max_samples:
            self.samples = self.samples[:max_samples]

        # Report class distribution
        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  ASVspoof 2019 LA Eval: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav = load_audio(self.audio_dir / f"{s['filename']}.flac", self.max_len)
        # Build attention mask: 1 for real audio, 0 for padding
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {
            'input_values': wav,
            'attention_mask': mask,
            'task_label': torch.tensor(s['label'], dtype=torch.long),
        }


class ASVspoof2021Dataset(Dataset):
    """ASVspoof 2021 LA/DF evaluation set (Yamagishi et al., 2021).

    The 2021 challenge used a different protocol format. Ground truth keys
    are available post-challenge in the keys/ directory. The trial_metadata.txt
    format varies slightly between LA and DF tracks, but both contain the
    audio key and a label column that includes 'bonafide' for genuine speech.

    Audio format: FLAC or WAV depending on track/version.
    """

    def __init__(self, keys_path, audio_dir, max_len=64000, max_samples=None):
        self.audio_dir = Path(audio_dir)
        self.max_len = max_len
        self.samples = []

        with open(keys_path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) < 2:
                    continue
                # Audio key is in column 1 (0-indexed); label determined by
                # presence of 'bonafide' anywhere in the line
                filename = p[1]
                label = 1 if 'bonafide' in line.lower() else 0
                self.samples.append({'filename': filename, 'label': label})

        if max_samples:
            self.samples = self.samples[:max_samples]

        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  ASVspoof 2021 Eval: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        # Try FLAC first, fall back to WAV (format varies by track)
        path = self.audio_dir / f"{s['filename']}.flac"
        if not path.exists():
            path = self.audio_dir / f"{s['filename']}.wav"
        wav = load_audio(path, self.max_len)
        mask = torch.ones(self.max_len, dtype=torch.long)
        return {
            'input_values': wav,
            'attention_mask': mask,
            'task_label': torch.tensor(s['label'], dtype=torch.long),
        }


class InTheWildDataset(Dataset):
    """In-the-Wild dataset (M\u00fcller et al., 2024).

    CSV metadata with columns: file, label.
    Labels: 'bona-fide'/'bonafide'/'real' -> 1, everything else -> 0.
    Audio: various formats (WAV, FLAC, MP3) in the release directory.

    This is the primary out-of-domain benchmark. The extreme domain shift
    (lab-trained model evaluated on real-world audio) is exactly what GLADA's
    domain adaptation is designed to address (Ganin et al., JMLR 2016).
    """
    BONAFIDE = {'bona-fide', 'bonafide', 'real', '1', 'true'}

    def __init__(self, metadata_path, audio_dir, max_len=64000,
                 use_labels=True, max_samples=None):
        self.audio_dir = Path(audio_dir)
        self.max_len = max_len
        self.use_labels = use_labels
        self.samples = []

        with open(metadata_path) as f:
            header = next(f).strip().lower().split(',')
            # Find label column by name (handles slight CSV variations)
            label_idx = next((i for i, c in enumerate(header)
                              if c in ['label', 'class']), -1)
            for line in f:
                p = line.strip().split(',')
                raw = p[label_idx].strip().lower() if label_idx >= 0 else 'spoof'
                label = 1 if raw in self.BONAFIDE else 0
                self.samples.append({'filename': p[0], 'label': label})

        if max_samples:
            self.samples = self.samples[:max_samples]

        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  In-the-Wild: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav = load_audio(self.audio_dir / s['filename'], self.max_len)
        return {
            'input_values': wav,
            'attention_mask': torch.ones(self.max_len, dtype=torch.long),
            'task_label': torch.tensor(
                s['label'] if self.use_labels else 0, dtype=torch.long),
        }


class WaveFakeDataset(Dataset):
    """WaveFake dataset (Frank & Sch\u00f6nherr, 2021).

    Directory structure:
      root/
        LJSpeech-1.1/wavs/          -> bonafide (original LJSpeech recordings)
        ljspeech_melgan/             -> spoof (MelGAN vocoder)
        ljspeech_full_band_melgan/   -> spoof (Full-band MelGAN)
        ljspeech_multi_band_melgan/  -> spoof (Multi-band MelGAN)
        ljspeech_hifiGAN/            -> spoof (HiFi-GAN)
        ljspeech_waveglow/           -> spoof (WaveGlow)
        ljspeech_parallel_wavegan/   -> spoof (Parallel WaveGAN)

    Bonafide audio is from LJSpeech-1.1 (Ito & Johnson, 2017), a single-speaker
    English audiobook corpus. All audio is 22,050 Hz natively; resampled to
    16,000 Hz by load_audio().

    This dataset tests generalization to diverse neural vocoders — a different
    type of distribution shift than ITW (vocoder artifacts vs. real-world noise).
    """

    def __init__(self, root_dir, max_len=64000, max_samples=None):
        self.root_dir = Path(root_dir)
        self.max_len = max_len
        self.samples = []
        self.vocoder_counts = {}  # Track per-vocoder sample counts

        # Scan for bonafide (LJSpeech-1.1/wavs/)
        bonafide_dir = self.root_dir / 'LJSpeech-1.1' / 'wavs'
        if bonafide_dir.exists():
            wav_files = sorted(bonafide_dir.glob('*.wav'))
            for f in wav_files:
                self.samples.append({
                    'path': f, 'label': 1, 'vocoder': 'bonafide'
                })
            self.vocoder_counts['bonafide'] = len(wav_files)

        # Scan for spoof (ljspeech_*/ subdirectories)
        for subdir in sorted(self.root_dir.iterdir()):
            if not subdir.is_dir():
                continue
            # Skip bonafide directory and hidden dirs
            if subdir.name.startswith('.') or subdir.name == 'LJSpeech-1.1':
                continue
            # Each ljspeech_* directory is a different vocoder
            vocoder_name = subdir.name.replace('ljspeech_', '')
            wav_files = sorted(list(subdir.rglob('*.wav')))
            for f in wav_files:
                self.samples.append({
                    'path': f, 'label': 0, 'vocoder': vocoder_name
                })
            self.vocoder_counts[vocoder_name] = len(wav_files)

        if max_samples:
            self.samples = self.samples[:max_samples]

        # Report per-vocoder breakdown
        n_bona = sum(1 for s in self.samples if s['label'] == 1)
        n_spoof = len(self.samples) - n_bona
        print(f'  WaveFake: {len(self.samples)} samples '
              f'({n_bona} bonafide, {n_spoof} spoof)')
        print(f'  Per-vocoder breakdown:')
        for voc, count in sorted(self.vocoder_counts.items()):
            label_str = 'bonafide' if voc == 'bonafide' else 'spoof'
            print(f'    {voc:30s} {count:6d} ({label_str})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        wav = load_audio(s['path'], self.max_len)
        return {
            'input_values': wav,
            'attention_mask': torch.ones(self.max_len, dtype=torch.long),
            'task_label': torch.tensor(s['label'], dtype=torch.long),
        }


def create_eval_dataloaders(config, max_samples=None):
    """Create DataLoaders for all available evaluation datasets.

    Only creates loaders for datasets whose paths exist on disk.
    Returns a dict mapping dataset name -> DataLoader.

    Args:
        config: GLADAConfig instance
        max_samples: Optional cap on samples per dataset (for debugging)

    Returns:
        Dict[str, DataLoader] keyed by dataset name
    """
    loaders = {}
    max_len = config.max_audio_len
    bs = config.batch_size * 2  # Use full batch (no source/target split for eval)

    # ASVspoof 2019 LA Eval
    if os.path.exists(ASV19_EVAL_PROTO) and os.path.exists(ASV19_EVAL_AUDIO):
        ds = ASVspoofEvalDataset(
            ASV19_EVAL_PROTO, ASV19_EVAL_AUDIO,
            max_len=max_len, max_samples=max_samples)
        loaders['ASV19-LA-Eval'] = DataLoader(
            ds, batch_size=bs, shuffle=False,
            num_workers=config.num_workers, pin_memory=True)
    else:
        print('  [SKIP] ASVspoof 2019 LA Eval: paths not found')

    # ASVspoof 2021 LA Eval
    if os.path.exists(ASV21_LA_KEYS) and os.path.exists(ASV21_LA_AUDIO):
        ds = ASVspoof2021Dataset(
            ASV21_LA_KEYS, ASV21_LA_AUDIO,
            max_len=max_len, max_samples=max_samples)
        loaders['ASV21-LA-Eval'] = DataLoader(
            ds, batch_size=bs, shuffle=False,
            num_workers=config.num_workers, pin_memory=True)
    else:
        print('  [SKIP] ASVspoof 2021 LA Eval: paths not found')

    # ASVspoof 2021 DF Eval
    if os.path.exists(ASV21_DF_KEYS) and os.path.exists(ASV21_DF_AUDIO):
        ds = ASVspoof2021Dataset(
            ASV21_DF_KEYS, ASV21_DF_AUDIO,
            max_len=max_len, max_samples=max_samples)
        loaders['ASV21-DF-Eval'] = DataLoader(
            ds, batch_size=bs, shuffle=False,
            num_workers=config.num_workers, pin_memory=True)
    else:
        print('  [SKIP] ASVspoof 2021 DF Eval: paths not found')

    # In-the-Wild
    if os.path.exists(ITW_META) and os.path.exists(ITW_AUDIO):
        ds = InTheWildDataset(
            ITW_META, ITW_AUDIO,
            max_len=max_len, use_labels=True, max_samples=max_samples)
        loaders['ITW'] = DataLoader(
            ds, batch_size=bs, shuffle=False,
            num_workers=config.num_workers, pin_memory=True)
    else:
        print('  [SKIP] In-the-Wild: paths not found')

    # WaveFake
    if os.path.exists(WAVEFAKE_ROOT):
        ds = WaveFakeDataset(
            WAVEFAKE_ROOT, max_len=max_len, max_samples=max_samples)
        if len(ds) > 0:
            loaders['WaveFake'] = DataLoader(
                ds, batch_size=bs, shuffle=False,
                num_workers=config.num_workers, pin_memory=True)
        else:
            print('  [SKIP] WaveFake: no samples found in directory')
    else:
        print('  [SKIP] WaveFake: root directory not found')

    print(f'\nCreated {len(loaders)} evaluation dataloaders: {list(loaders.keys())}')
    return loaders

## Model Architecture

> **Rule 13 Note**: In the `glada_experiment.ipynb` training notebook, all model logic lives in the `glada` package (imported via `from glada.models import GLADAModel`). This evaluation notebook also imports from the package for **new-style GLADA checkpoints** (see the "Load Trained Model" section below). The architecture classes reproduced in the next code cell are **only for legacy checkpoint compatibility** — they allow loading checkpoints that were saved before the modular package existed. Do not modify this code block; it must remain bit-for-bit identical to the architecture that produced the checkpoints.

The complete GLADA architecture must be defined identically to the training configuration to ensure checkpoint compatibility. All component classes are reproduced here for legacy loading only.

**Architecture overview** (see `glada_experiment.ipynb` Section 2 and `Coding/glada/models/` for detailed component-by-component documentation):

```
Raw Audio (16kHz, 4s)
  -> SSL Backbone (unfrozen, gradient checkpointing)
  -> Per-Layer LayerNorm (El-Kheir et al., NAACL 2025)
  -> Grouped Hierarchical Attention Pooling (3 groups of layers)
       |-> Group 1 (layers 1-8):  ASP + group embedding
       |-> Group 2 (layers 9-16): ASP + group embedding
       |-> Group 3 (layers 17-24): ASP + group embedding
       |-> CKA diversity loss (Tran et al., MM 2025)
  -> VIB Bottleneck (Alemi et al., ICLR 2017)
  -> Task Classifier (bonafide vs. spoof)
  -> Speaker Adversary via GRL (Ganin et al., JMLR 2016)
  -> Per-Group Domain Discriminators via GRL [GLADA novel contribution]
```

**Loss function**:
$$\mathcal{L} = \mathcal{L}_{\text{spoof}} + \alpha \mathcal{L}_{\text{GLADA}} + \beta \mathcal{L}_{\text{KL}} + \gamma \mathcal{L}_{\text{speaker}} + \delta \mathcal{L}_{\text{CKA}}$$

In [30]:
# ============================================================================
# MODEL ARCHITECTURE — Complete GLADA model (must match training notebook)
# All classes reproduced verbatim for checkpoint compatibility.
# ============================================================================

# ----------------------------------------------------------------------------
# 1. Gradient Reversal Layer (Ganin et al., JMLR 2016)
# Forward pass: identity. Backward pass: negate gradient * alpha.
# This is the core mechanism for domain-adversarial training — the backbone
# learns features that are *useful* for the task but *uninformative* for the
# domain discriminator, because the discriminator's gradient is reversed.
# ----------------------------------------------------------------------------

class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)  # Identity in forward pass

    @staticmethod
    def backward(ctx, grad_output):
        # Negate gradient and scale by alpha (GRL strength)
        return grad_output.neg() * ctx.alpha, None


# ----------------------------------------------------------------------------
# 2. Per-Layer Normalization
# Motivation: El-Kheir et al. (NAACL 2025) showed SSL layers differ in
# magnitude and encode fundamentally different deepfake information.
# LayerNorm per hidden state prevents gradient dominance by high-magnitude layers.
# ----------------------------------------------------------------------------

class PerLayerNorm(nn.Module):
    """Apply independent LayerNorm to each SSL hidden state before aggregation.
    Normalizes magnitude across layers to prevent gradient dominance."""

    def __init__(self, num_layers, hidden_dim):
        super().__init__()
        # One LayerNorm per SSL layer (including CNN embedding layer)
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])

    def forward(self, hidden_states):
        """hidden_states: list of (B, T, D) tensors"""
        return [norm(h) for norm, h in zip(self.norms, hidden_states)]


# ----------------------------------------------------------------------------
# 3. Attentive Statistics Pooling (Okabe et al., Interspeech 2018)
# Instead of mean-pooling over time (which treats all frames equally), ASP
# learns attention weights over the temporal dimension, then computes a
# weighted mean AND weighted standard deviation. Output = 2 * input_dim.
# This has become standard in speaker verification and is used by most
# top-performing deepfake detection systems.
# ----------------------------------------------------------------------------

class AttentiveStatisticsPooling(nn.Module):
    """Attentive Statistics Pooling: weighted mean + weighted std."""

    def __init__(self, input_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 128),  # Bottleneck attention
            nn.Tanh(),
            nn.Linear(128, 1),           # Scalar attention per frame
        )

    def forward(self, x):
        """x: (B, T, D) -> (B, 2*D)"""
        w = F.softmax(self.attention(x), dim=1)      # (B, T, 1) attention weights
        mu = (x * w).sum(dim=1)                       # (B, D) weighted mean
        sigma = torch.sqrt(((x - mu.unsqueeze(1))**2 * w).sum(dim=1) + 1e-6)  # (B, D) weighted std
        return torch.cat([mu, sigma], dim=1)          # (B, 2*D)


# ----------------------------------------------------------------------------
# 4. Linear CKA (Kornblith et al., ICML 2019)
# Centered Kernel Alignment measures representational similarity between two
# feature matrices. Used as a diversity loss (Tran et al., ACM MM 2025) to
# encourage different layer groups to learn complementary features.
# Low CKA = groups capture different information -> more diverse representations.
# ----------------------------------------------------------------------------

def compute_cka(X, Y):
    """Linear CKA between two representation matrices.
    X, Y: (B, D) feature matrices. Returns scalar similarity in [0, 1]."""
    # Center the representations (zero mean)
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    # HSIC computation via linear kernel
    hsic_xy = (X @ X.T * (Y @ Y.T)).sum()
    hsic_xx = (X @ X.T * (X @ X.T)).sum()
    hsic_yy = (Y @ Y.T * (Y @ Y.T)).sum()
    return hsic_xy / (torch.sqrt(hsic_xx * hsic_yy) + 1e-8)


# ----------------------------------------------------------------------------
# 5. Hierarchical Attention Pooling + CKA Diversity
# Inspired by HierCon (Liang et al., 2026, arXiv 2602.01032) which groups
# SSL layers and applies contrastive learning within groups.
# GLADA extends this with: (a) learnable softmax attention over layers in
# each group, (b) ASP pooling per group, (c) CKA diversity loss between
# groups, and (d) per-group domain discriminators (the novel contribution).
#
# Group assignment: layers distributed evenly across K groups.
# E.g., WavLM-Large has 25 hidden states (24 transformer + 1 CNN), K=4 groups:
#   Group 0: layers 0-5 (low-level acoustic features)
#   Group 1: layers 6-11 (phonetic/prosodic features)
#   Group 2: layers 12-17 (linguistic features)
#   Group 3: layers 18-24 (task-specific features, gets remainder)
# ----------------------------------------------------------------------------

class HierarchicalAttentionPooling(nn.Module):
    """Group SSL hidden states into K groups.
    Per-group: learnable softmax-weighted layer sum -> ASP.
    Concatenate group outputs -> linear projection.
    CKA diversity loss between group representations."""

    def __init__(self, num_hidden_states, hidden_dim, num_groups, output_dim):
        super().__init__()
        self.num_groups = num_groups
        self.hidden_dim = hidden_dim

        # Assign layers to groups (evenly, last group gets remainder)
        layers_per_group = num_hidden_states // num_groups
        self.group_assignments = []
        for g in range(num_groups):
            start = g * layers_per_group
            end = start + layers_per_group if g < num_groups - 1 else num_hidden_states
            self.group_assignments.append(list(range(start, end)))

        # Per-group learnable layer weights (softmax-normalized in forward)
        self.group_weights = nn.ParameterList([
            nn.Parameter(torch.ones(len(layers)) / len(layers))
            for layers in self.group_assignments
        ])

        # Per-group ASP (Okabe et al., 2018)
        self.group_asp = nn.ModuleList([
            AttentiveStatisticsPooling(hidden_dim)
            for _ in range(num_groups)
        ])

        # Projection: concat of K groups * 2*D -> output_dim
        self.projection = nn.Linear(num_groups * 2 * hidden_dim, output_dim)

    def forward(self, hidden_states):
        """hidden_states: list of (B, T, D) tensors (already layer-normed)
        Returns: pooled (B, output_dim), cka_loss scalar, group_outputs list of (B, 2*D)"""
        group_outputs = []

        for g, (layer_ids, weights) in enumerate(
                zip(self.group_assignments, self.group_weights)):
            # Stack layers in this group and compute softmax-weighted sum
            layers = torch.stack([hidden_states[i] for i in layer_ids])  # (G_k, B, T, D)
            w = F.softmax(weights, dim=0).view(-1, 1, 1, 1)             # (G_k, 1, 1, 1)
            group_feat = (layers * w).sum(dim=0)       # (B, T, D)
            group_pooled = self.group_asp[g](group_feat)  # (B, 2*D) via ASP
            group_outputs.append(group_pooled)

        # Concatenate all group outputs and project
        concat = torch.cat(group_outputs, dim=1)       # (B, K * 2*D)
        pooled = self.projection(concat)                # (B, output_dim)

        # CKA diversity loss: penalize high similarity between groups
        # Lower CKA = groups learn more diverse/complementary features
        cka_loss = torch.tensor(0.0, device=pooled.device)
        n_pairs = 0
        for i in range(len(group_outputs)):
            for j in range(i + 1, len(group_outputs)):
                cka_loss = cka_loss + compute_cka(group_outputs[i], group_outputs[j])
                n_pairs += 1
        if n_pairs > 0:
            cka_loss = cka_loss / n_pairs  # Mean pairwise CKA

        return pooled, cka_loss, group_outputs


# ----------------------------------------------------------------------------
# 6. Grouped-Layer Domain Discriminators (NOVEL -- GLADA contribution)
# Precedents: MLADA (Qu et al., Neurocomputing 2022) for multi-layer DA in
# vision; DAA (Ryu et al., ACL Findings 2021) for per-layer adversarial
# adapters on BERT. El-Kheir et al. (NAACL 2025) provides empirical
# motivation: different SSL layers encode different deepfake artifacts.
#
# GLADA applies separate GRL + domain discriminators at each layer group's
# ASP output. This enforces domain invariance at every abstraction level of
# the SSL hierarchy, rather than only at the final pooled representation.
# ----------------------------------------------------------------------------

class GroupedLayerDomainDiscriminators(nn.Module):
    """Per-group domain discriminators for Grouped-Layer Adversarial DA (GLADA).
    Each layer group's ASP output gets its own GRL + lightweight domain discriminator.
    This is the primary novel contribution: enforcing domain invariance at each
    abstraction level of the SSL representation hierarchy."""

    def __init__(self, num_groups, input_dim, hidden_dim=128):
        super().__init__()
        # One lightweight MLP per group: input_dim -> hidden -> hidden -> 2 (domain classes)
        self.discriminators = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(hidden_dim, 2),  # Binary: source vs target domain
            )
            for _ in range(num_groups)
        ])

    def forward(self, group_outputs, grl_alpha):
        """Args:
            group_outputs: list of (B, 2*D) tensors from per-group ASP
            grl_alpha: float, GRL reversal strength
        Returns:
            list of (B, 2) domain logits per group"""
        logits = []
        for disc, feat in zip(self.discriminators, group_outputs):
            # Apply GRL before each discriminator — reversed gradient flows
            # back through the pooling and into the backbone, encouraging
            # domain-invariant features at this layer group's abstraction level
            feat_rev = GradientReversal.apply(feat, grl_alpha)
            logits.append(disc(feat_rev))
        return logits


# ----------------------------------------------------------------------------
# 7. Variational Information Bottleneck (Alemi et al., ICLR 2017)
# VIB compresses the representation h into a stochastic bottleneck z ~ N(mu, sigma^2).
# The KL term KL(q(z|x) || N(0,I)) acts as a regularizer that removes
# task-irrelevant information. Used by IB-CAAN (Huang et al., 2025) for
# deepfake detection. At eval time, z = mu (deterministic, no sampling noise).
# ----------------------------------------------------------------------------

class VIBBottleneck(nn.Module):
    """Variational Information Bottleneck.
    Maps pooled features to (mu, logvar), samples z, computes KL loss."""

    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()
        # Separate heads for mean and log-variance
        self.mu_head = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.GELU(),
            nn.Linear(input_dim, bottleneck_dim),
        )
        self.logvar_head = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.GELU(),
            nn.Linear(input_dim, bottleneck_dim),
        )

    def forward(self, x, training=True):
        """x: (B, D) -> z: (B, bottleneck_dim), kl_loss: scalar"""
        mu = self.mu_head(x)
        logvar = self.logvar_head(x)

        if training:
            # Reparameterization trick: z = mu + eps * std
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            z = mu + std * eps
        else:
            # Deterministic at eval time (no sampling noise)
            z = mu

        # KL divergence: KL(q(z|x) || N(0,I))
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return z, kl_loss


# ----------------------------------------------------------------------------
# 8. GLADAModel — Full model integrating all components
# Forward pass:
#   1. Backbone: extract all hidden states (output_hidden_states=True)
#   2. Per-Layer Norm: normalize each hidden state independently
#   3. Hierarchical Pooling: group layers, learn attention weights, ASP per group
#   4. VIB Bottleneck: compress to z (stochastic during training, deterministic at eval)
#   5. Task Classifier: binary spoof/bonafide prediction
#   6. (Optional) Global domain disc, GLADA per-group domain discs, speaker adversary
# ----------------------------------------------------------------------------

class GLADAModel(nn.Module):
    def __init__(self, config: GLADAConfig):
        super().__init__()
        self.config = config

        # --- Backbone: pre-trained SSL model ---
        self.backbone = AutoModel.from_pretrained(config.hf_model_id)
        # Freeze CNN encoder permanently (it's a fixed feature extractor)
        if hasattr(self.backbone, 'feature_extractor'):
            self.backbone.feature_extractor._freeze_parameters()
        elif hasattr(self.backbone, 'feature_projection'):
            for p in self.backbone.feature_projection.parameters():
                p.requires_grad = False

        # --- Per-layer normalization (El-Kheir et al., NAACL 2025 motivated) ---
        self.layer_norms = PerLayerNorm(config.total_hidden_states, config.hidden_dim)

        # --- Hierarchical Attention Pooling (HierCon-inspired, Liang et al. 2026) ---
        self.pooling = HierarchicalAttentionPooling(
            num_hidden_states=config.total_hidden_states,
            hidden_dim=config.hidden_dim,
            num_groups=config.num_groups,
            output_dim=config.hidden_dim,
        )

        # --- VIB Bottleneck or deterministic projection (Alemi et al. 2017) ---
        if config.use_vib:
            self.vib = VIBBottleneck(config.hidden_dim, config.bottleneck_dim)
        else:
            self.vib = None
            self.deterministic_proj = nn.Sequential(
                nn.Linear(config.hidden_dim, config.hidden_dim),
                nn.GELU(),
                nn.Linear(config.hidden_dim, config.bottleneck_dim),
            )

        # --- Task Classifier (spoof detection) ---
        self.task_classifier = nn.Sequential(
            nn.Linear(config.bottleneck_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2),  # Binary: spoof (0) vs bonafide (1)
        )

        # --- Speaker Adversary (deeper, 3-layer) ---
        if config.use_speaker_adversary:
            self.speaker_adversary = nn.Sequential(
                nn.Linear(config.bottleneck_dim, 512),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(256, config.num_speakers),  # 107 ASVspoof speakers
            )
        else:
            self.speaker_adversary = None

        # --- Global Domain Discriminator (baseline, off by default) ---
        if config.use_global_domain_disc:
            self.domain_disc = nn.Sequential(
                nn.Linear(config.bottleneck_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(256, 2),
            )
        else:
            self.domain_disc = None

        # --- GLADA: Per-group Domain Discriminators (NOVEL) ---
        if config.use_glada:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=config.num_groups,
                input_dim=2 * config.hidden_dim,  # ASP output: mean + std
                hidden_dim=config.glada_disc_hidden,
            )
        else:
            self.glada_disc = None

        # GRL alphas (set by trainer each epoch during training)
        self.domain_grl_alpha = 0.0
        self.speaker_grl_alpha = 0.0

    def freeze_backbone(self):
        """Freeze all backbone parameters (Phase 1: train heads only)."""
        for p in self.backbone.parameters():
            p.requires_grad = False
        if hasattr(self.backbone, 'feature_extractor'):
            self.backbone.feature_extractor._freeze_parameters()

    def unfreeze_backbone(self):
        """Unfreeze backbone for Phase 2 fine-tuning with gradient checkpointing."""
        for p in self.backbone.parameters():
            p.requires_grad = True
        # Always keep CNN frozen (it's a fixed feature extractor)
        if hasattr(self.backbone, 'feature_extractor'):
            self.backbone.feature_extractor._freeze_parameters()
        # Enable gradient checkpointing to reduce memory (O(sqrt(n)) instead of O(n))
        self.backbone.gradient_checkpointing_enable()

    def forward(self, input_values, attention_mask=None):
        # 1. Backbone with all hidden states
        outputs = self.backbone(
            input_values,
            attention_mask=attention_mask,
            output_hidden_states=True,  # Returns all layer outputs
        )
        hidden_states = list(outputs.hidden_states)  # List of (B, T, D)

        # 2. Per-layer normalization
        hidden_states = self.layer_norms(hidden_states)

        # 3. Hierarchical pooling + CKA loss + per-group outputs
        pooled, cka_loss, group_outputs = self.pooling(hidden_states)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=pooled.device)

        # 4. VIB or deterministic projection
        if self.vib is not None:
            z, kl_loss = self.vib(pooled, training=self.training)
        else:
            z = self.deterministic_proj(pooled)
            kl_loss = torch.tensor(0.0, device=pooled.device)

        # 5. Task classifier (spoof/bonafide)
        spoof_logits = self.task_classifier(z)

        result = {
            "spoof_logits": spoof_logits,
            "kl_loss": kl_loss,
            "cka_loss": cka_loss,
            "features": z,
        }

        # 6. Global domain discriminator (baseline, off by default)
        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result["domain_logits"] = self.domain_disc(z_domain)

        # 7. GLADA: per-group domain discriminators (NOVEL)
        if self.glada_disc is not None:
            result["glada_domain_logits"] = self.glada_disc(group_outputs, self.domain_grl_alpha)
            result["group_outputs"] = group_outputs

        # 8. Speaker adversary
        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result["speaker_logits"] = self.speaker_adversary(z_speaker)

        return result


print("Model architecture defined. All components match training notebook.")

Model architecture defined. All components match training notebook.


## Load Trained Model

Load a trained GLADA checkpoint. The checkpoint contains:
- `model`: model state dict (saved via `model.state_dict()`)
- `epoch`: training epoch at which the best EER was achieved
- `best_eer`: best validation EER during training
- `config`: full `GLADAConfig` as a dict for reproducibility

The `load_model` function reconstructs the config from the checkpoint's saved values, ensuring that architecture parameters (backbone, hidden_dim, num_groups, etc.) match exactly what was used during training. This is critical -- a mismatch in any of these causes a `state_dict` loading error.

In [32]:
# ============================================================================
# Load trained GLADA model from checkpoint
# Checkpoint format: {'model': state_dict, 'epoch': int, 'best_eer': float,
#                      'config': dict}  (saved by training loop)
# ============================================================================

def load_model(checkpoint_path, backbone='wavlm-large'):
    """Load trained GLADA model from checkpoint.

    Args:
        checkpoint_path: Path to .pt file with model state_dict + config
        backbone: Backbone key (used as fallback if config not in checkpoint)

    Returns:
        model: GLADAModel in eval mode
        config: GLADAConfig reconstructed from checkpoint
    """
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Reconstruct config from checkpoint (ensures architecture match)
    saved_config = ckpt.get('config', {})
    config = GLADAConfig(backbone=backbone)
    # Override with saved config values to match training exactly
    for k, v in saved_config.items():
        if hasattr(config, k):
            setattr(config, k, v)
    config.__post_init__()  # Recompute derived values (hf_model_id, hidden_dim, etc.)

    # Build model and load weights
    model = GLADAModel(config)
    model.load_state_dict(ckpt['model'])
    model.eval()  # Set to eval mode (deterministic VIB, no dropout)

    # Print checkpoint info
    epoch = ckpt.get('epoch', '?')
    best_eer = ckpt.get('best_eer', '?')
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Loaded from epoch {epoch}, best EER: {best_eer}")
    print(f"Backbone: {config.backbone} ({config.hf_model_id})")
    print(f"Parameters: {n_params:.1f}M")
    print(f"Components: GLADA={config.use_glada}, VIB={config.use_vib}, "
          f"CKA={config.use_cka_loss}, Speaker={config.use_speaker_adversary}")
    return model, config


# ============================================================================
# CONFIGURE CHECKPOINT PATH HERE
# Update the path below to point to your trained model checkpoint.
# Expected location: /content/drive/MyDrive/Jakob/Experiments/<experiment>/<run_id>/best_model.pt
# ============================================================================

CHECKPOINT = '/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/20260407_032408/best_model.pt'  # <-- UPDATE THIS
BACKBONE = 'wavlm-large'  # Must match training backbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model, config = load_model(CHECKPOINT, BACKBONE)
model = model.to(device)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB


Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Loaded from epoch 1, best EER: 11.34499954740237
Backbone: wavlm-large (microsoft/wavlm-large)
Parameters: 328.4M
Components: GLADA=True, VIB=True, CKA=True, Speaker=True


### Legacy Model Loading (DANN-IB from Survey Paper)

The preliminary experiments in Section VIII of the survey paper used `DomainAdaptationModel`
from `domain_adaptation_e2e.py` — a simpler architecture than GLADA (frozen SSL backbone +
linear classifier + optional domain discriminator). To re-evaluate those checkpoints on all
5 standard datasets, we need to reconstruct the legacy model class.

**When to use this**: Set `USE_LEGACY = True` below if evaluating old DANN-IB/CAAN/MMD/CORAL
checkpoints from `/content/drive/MyDrive/Jakob/Experiments/` (configs matching the survey paper
Tables III-V).

**Configs to re-evaluate** (from survey paper):
- WavLM-Base-Plus + DANN-IB (seeds 42, 123, 456)
- WavLM-Base-Plus + CAAN (seeds 42, 123, 456)
- WavLM-Large + CAAN (seeds 42, 123, 456)
- WavLM-Large + DANN-IB (seeds 42, 123, 456)
- XLS-R 300M + DANN-IB (seeds 42, 123, 456)


In [33]:
# ============================================================================
# LEGACY MODEL LOADING (DANN-IB / Survey Paper Experiments)
# ============================================================================
# The old DomainAdaptationModel uses a frozen SSL backbone with mean pooling
# and a linear classifier head. Much simpler than GLADA.

USE_LEGACY = False  # Set True to evaluate old DANN-IB checkpoints

class LegacyGradientReversal(torch.autograd.Function):
    """GRL from domain_adaptation_e2e.py."""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None


class LegacyDomainAdaptationModel(nn.Module):
    """Reconstructed from domain_adaptation_e2e.py for checkpoint loading.
    Architecture: frozen SSL -> mean pool -> classifier head.
    Optional: domain discriminator with GRL."""

    def __init__(self, model_name='microsoft/wavlm-base-plus', hidden_size=768,
                 has_domain_disc=True):
        super().__init__()
        self.wavlm = AutoModel.from_pretrained(model_name)
        self.wavlm.feature_extractor._freeze_parameters()
        for p in self.wavlm.parameters():
            p.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        )
        self.domain_disc = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.LayerNorm(256),
            nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 2)
        ) if has_domain_disc else None
        self.grl_alpha = 0.0

    def forward(self, input_values, attention_mask=None):
        features = self.wavlm(
            input_values, attention_mask=attention_mask
        ).last_hidden_state.mean(1)
        result = {
            'spoof_logits': self.classifier(features),
            'features': features
        }
        if self.domain_disc is not None:
            result['domain_logits'] = self.domain_disc(
                LegacyGradientReversal.apply(features, self.grl_alpha)
            )
        return result


# --- Legacy model name mapping ---
LEGACY_BACKBONE_MAP = {
    'wavlm-base-plus': ('microsoft/wavlm-base-plus', 768),
    'wavlm-large': ('microsoft/wavlm-large', 1024),
    'xlsr-300m': ('facebook/wav2vec2-xls-r-300m', 1024),
    'wav2vec2-base': ('facebook/wav2vec2-base-960h', 768),
    'wav2vec2-large': ('facebook/wav2vec2-large-960h', 1024),
}


def load_legacy_model(checkpoint_path, backbone='wavlm-base-plus'):
    """Load a DomainAdaptationModel checkpoint from old experiments.

    The old checkpoints save state_dict with keys like 'wavlm.encoder...',
    'classifier.0.weight', 'domain_disc.0.weight'. The key 'task_logits'
    in the old model maps to 'spoof_logits' in evaluation code.
    """
    model_name, hidden = LEGACY_BACKBONE_MAP[backbone]
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

    # Check if domain_disc keys exist in state_dict
    state = ckpt['model']
    has_disc = any('domain_disc' in k for k in state.keys())

    model = LegacyDomainAdaptationModel(
        model_name=model_name, hidden_size=hidden, has_domain_disc=has_disc
    )
    model.load_state_dict(state)
    model.eval()

    epoch = ckpt.get('epoch', '?')
    best_eer = ckpt.get('best_eer', ckpt.get('eer', '?'))
    print(f'Legacy model loaded: {backbone} (epoch {epoch}, EER {best_eer})')
    print(f'  Domain disc: {has_disc}')
    return model


# --- Override model if using legacy ---
if USE_LEGACY:
    LEGACY_CHECKPOINT = '/content/drive/MyDrive/Jakob/Experiments/dann_ib_wavlm_bp_seed42/best_model.pt'  # Edit path
    LEGACY_BACKBONE = 'wavlm-base-plus'  # Edit backbone
    model = load_legacy_model(LEGACY_CHECKPOINT, LEGACY_BACKBONE)
    model = model.to(device)
    print('Using LEGACY model for evaluation')


## Multi-Dataset Evaluator

> **Rule 13 Note**: The canonical evaluator lives in `glada.evaluation.evaluator.MultiDatasetEvaluator`. The class defined below is a **simplified inline version** kept for standalone portability. For new experiments, prefer `from glada.evaluation.evaluator import MultiDatasetEvaluator`.

Computes **Equal Error Rate (EER)** with bootstrap 95% confidence intervals (1000 resamples):

$$\text{EER} = \frac{\text{FAR}(\tau^*) + \text{FRR}(\tau^*)}{2}, \quad \tau^* = \arg\min_\tau |\text{FAR}(\tau) - \text{FRR}(\tau)|$$

EER is the primary metric for audio deepfake detection, standardized by the ASVspoof challenge series (Yamagishi et al., 2021). It is threshold-independent: FAR = FRR at the operating point, so neither false alarms nor missed detections dominate the score. Lower EER is better; 50% = random chance.

Also computes AUC and accuracy at the EER threshold for completeness.

### Bootstrap Confidence Intervals

Following Rule 3 (committee-ready quality), we report 95% bootstrap CIs using 1000 resamples (Efron & Tibshirani, 1993):

$$\text{CI}_{95\%} = [\text{percentile}(\text{EER}_b, 2.5), \text{percentile}(\text{EER}_b, 97.5)], \quad b \in 1..1000$$

This provides statistical rigor beyond point estimates — critical for claiming that GLADA outperforms baselines when EER differences may be < 1%.

### Design

1. Takes a pre-loaded model (not a checkpoint path) — decouples loading from evaluation, enabling multi-checkpoint comparison without re-defining the class
2. Runs inference with AMP (bfloat16/float16) to match training-time precision
3. **BFloat16 fix**: `probs.float().cpu().numpy()` — bfloat16 tensors cannot be directly converted to numpy; must cast to float32 first
4. Reports results as both markdown table (notebook display) and LaTeX row (thesis copy-paste)

In [34]:
# ============================================================================
# Multi-Dataset Evaluator with Bootstrap CI
# Primary metric: EER (ASVspoof/ITW standard)
# Bootstrap: 1000 resamples for 95% CI (committee-ready statistical rigor)
# ============================================================================

class MultiDatasetEvaluator:
    """Evaluate a GLADA model on multiple datasets with bootstrap CI.

    Decoupled from checkpoint loading: takes a pre-loaded model so the same
    evaluator can be reused for multi-model comparison without redundant I/O."""

    def __init__(self, model, device, amp_dtype=torch.bfloat16):
        """
        Args:
            model: GLADAModel in eval mode
            device: torch.device for inference
            amp_dtype: AMP precision (bfloat16 for A100, float16 for T4)
        """
        self.model = model
        self.device = device
        self.amp_dtype = amp_dtype
        self.model.eval()

    @torch.no_grad()
    def evaluate_dataset(self, loader, name="dataset"):
        """Run inference on a single dataset and compute EER + bootstrap CI.

        Args:
            loader: DataLoader yielding dicts with 'input_values', 'attention_mask', 'task_label'
            name: Display name for logging

        Returns:
            dict with eer, auc, acc, ci_low, ci_high, n_samples
        """
        scores, labels = [], []

        # Forward pass with AMP (matches training precision)
        for batch in tqdm(loader, desc=f"Eval {name}"):
            inp = batch['input_values'].to(self.device, non_blocking=True)
            mask = batch['attention_mask'].to(self.device, non_blocking=True)
            with autocast('cuda', enabled=True, dtype=self.amp_dtype):
                out = self.model(inp, attention_mask=mask)
            # Bonafide probability as the score (class 1 = bonafide)
            probs = F.softmax(out['spoof_logits'], dim=1)[:, 1]
            scores.extend(probs.float().cpu().numpy())
            labels.extend(batch['task_label'].numpy())

        labels, scores = np.array(labels), np.array(scores)

        # Edge case: only one class present (e.g., all bonafide)
        if len(np.unique(labels)) < 2:
            return {
                "name": name, "eer": 50.0, "auc": 0.5, "acc": 0.0,
                "n_samples": len(labels), "ci_low": 50.0, "ci_high": 50.0
            }

        # --- EER computation ---
        # EER = threshold where FPR == FNR (false positive rate == false negative rate)
        fpr, tpr, thresholds = roc_curve(labels, scores)
        fnr = 1 - tpr
        eer_idx = np.nanargmin(np.abs(fpr - fnr))  # Index of EER crossing
        eer = (fpr[eer_idx] + fnr[eer_idx]) / 2 * 100  # Average of FPR/FNR at crossing

        # AUC (Area Under ROC Curve)
        auc = roc_auc_score(labels, scores)

        # Accuracy at EER threshold
        threshold = thresholds[eer_idx]
        preds = (scores >= threshold).astype(int)
        acc = (preds == labels).mean() * 100

        # --- Bootstrap 95% CI on EER (1000 resamples) ---
        n_bootstrap = 1000
        rng = np.random.RandomState(42)  # Reproducible bootstrap
        eer_samples = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(labels), len(labels), replace=True)
            b_labels, b_scores = labels[idx], scores[idx]
            if len(np.unique(b_labels)) < 2:
                continue  # Skip degenerate resamples
            b_fpr, b_tpr, _ = roc_curve(b_labels, b_scores)
            b_fnr = 1 - b_tpr
            b_eer_idx = np.nanargmin(np.abs(b_fpr - b_fnr))
            eer_samples.append((b_fpr[b_eer_idx] + b_fnr[b_eer_idx]) / 2 * 100)

        ci_low = np.percentile(eer_samples, 2.5) if eer_samples else eer
        ci_high = np.percentile(eer_samples, 97.5) if eer_samples else eer

        return {
            "name": name, "eer": round(eer, 2), "auc": round(auc, 4),
            "acc": round(acc, 1), "n_samples": len(labels),
            "ci_low": round(ci_low, 2), "ci_high": round(ci_high, 2),
        }

    def evaluate_all(self, datasets: dict):
        """Evaluate on all datasets and print results.

        Args:
            datasets: dict of {name: DataLoader}

        Returns:
            dict of {name: result_dict}
        """
        results = {}
        for name, loader in datasets.items():
            results[name] = self.evaluate_dataset(loader, name)
            r = results[name]
            print(f"  {name}: EER={r['eer']:.2f}% [{r['ci_low']:.2f}, {r['ci_high']:.2f}] "
                  f"AUC={r['auc']:.4f} Acc={r['acc']:.1f}% (n={r['n_samples']})")
        return results

    @staticmethod
    def results_to_markdown(results):
        """Format results as a markdown table for notebook display."""
        lines = [
            "| Dataset | EER (%) | 95% CI | AUC | Samples |",
            "|---------|---------|--------|-----|---------|",
        ]
        for name, r in results.items():
            lines.append(
                f"| {name} | {r['eer']:.2f} | "
                f"[{r.get('ci_low', r['eer']):.2f}, {r.get('ci_high', r['eer']):.2f}] | "
                f"{r['auc']:.4f} | {r['n_samples']:,} |")
        return "\n".join(lines)

    @staticmethod
    def results_to_latex(results):
        """Format results as a LaTeX table for the thesis manuscript.
        Follows Rule 9 guidelines: clean formatting, no unnecessary lines."""
        lines = [
            r"\begin{table}[h]",
            r"\centering",
            r"\caption{Cross-domain evaluation results. EER (\%) with 95\% bootstrap confidence intervals (1000 resamples).}",
            r"\label{tab:eval_results}",
            r"\begin{tabular}{lcccc}",
            r"\toprule",
            r"Dataset & EER (\%) & 95\% CI & AUC & Samples \\",
            r"\midrule",
        ]
        for name, r in results.items():
            lines.append(
                f"{name} & {r['eer']:.2f} & "
                f"[{r.get('ci_low', r['eer']):.2f}, {r.get('ci_high', r['eer']):.2f}] & "
                f"{r['auc']:.4f} & {r['n_samples']:,} \\\\")
        lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
        return "\n".join(lines)


print("MultiDatasetEvaluator defined.")

MultiDatasetEvaluator defined.


## Sequential Evaluation

Extracts one dataset at a time, evaluates, then deletes to stay within 70GB disk.

In [36]:
# ============================================================================
# Sequential Evaluation: extract -> eval -> cleanup -> next
# Fits within 70GB free-tier disk by only keeping one dataset at a time.
# ============================================================================

# Determine AMP dtype based on GPU
if device.type == 'cuda':
    gpu_name = torch.cuda.get_device_name(0).lower()
    amp_dtype = torch.bfloat16 if 'a100' in gpu_name or 'h100' in gpu_name else torch.float16
else:
    amp_dtype = torch.float32
print(f"AMP dtype: {amp_dtype}")

evaluator = MultiDatasetEvaluator(model, device, amp_dtype=amp_dtype)

all_results = {}
print("=" * 70)
print("SEQUENTIAL EVALUATION (extract -> eval -> cleanup)")
print("=" * 70)

for ds in DATASET_REGISTRY:
    name = ds['name']
    print(f"\n{'─' * 50}")
    print(f"Dataset: {name}")
    print(f"{'─' * 50}")

    # 1. Extract (if this dataset has its own archive)
    if ds.get('archive'):
        if ds['archive'] == 'multipart':
            # ASVspoof 2021 DF: concatenate split archives then extract
            ok = extract_multipart_dataset(
                '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part_*',
                ds['local_dir'], name)
        else:
            ok = extract_dataset(ds['archive'], ds['local_dir'], name)
        if not ok:
            print(f"  SKIPPING {name} (archive not found)")
            continue

    # 2. Create dataloader
    try:
        maker = ds['make_loader']
        if maker == 'auto_2021_LA':
            keys = find_keys_file(ds['local_dir'], 'LA')
            audio = find_audio_dir(ds['local_dir'], 'LA')
            if not keys or not audio:
                import subprocess
                tree = subprocess.run(['find', ds['local_dir'], '-maxdepth', '3', '-type', 'f'],
                                      capture_output=True, text=True)
                print(f"  Keys file: {keys}")
                print(f"  Audio dir: {audio}")
                print(f"  Extracted files (first 20):")
                for line in tree.stdout.strip().split('\n')[:20]:
                    print(f"    {line}")
                raise FileNotFoundError(f"Could not find keys or audio for 2021 LA")
            print(f"  Auto-discovered keys: {keys}")
            print(f"  Auto-discovered audio: {audio}")
            loader = DataLoader(
                ASVspoof2021Dataset(keys, audio, max_len=config.max_audio_len),
                batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        elif maker == 'auto_2021_DF':
            keys = find_keys_file(ds['local_dir'], 'DF')
            audio = find_audio_dir(ds['local_dir'], 'DF')
            if not keys or not audio:
                import subprocess
                tree = subprocess.run(['find', ds['local_dir'], '-maxdepth', '3', '-type', 'f'],
                                      capture_output=True, text=True)
                print(f"  Keys file: {keys}")
                print(f"  Audio dir: {audio}")
                print(f"  Extracted files (first 20):")
                for line in tree.stdout.strip().split('\n')[:20]:
                    print(f"    {line}")
                raise FileNotFoundError(f"Could not find keys or audio for 2021 DF")
            print(f"  Auto-discovered keys: {keys}")
            print(f"  Auto-discovered audio: {audio}")
            loader = DataLoader(
                ASVspoof2021Dataset(keys, audio, max_len=config.max_audio_len),
                batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        else:
            loader = maker()
        print(f"  Loaded {len(loader.dataset)} samples")
    except Exception as e:
        print(f"  ERROR creating dataloader for {name}: {e}")
        if ds.get('local_dir') and not ds.get('no_cleanup'):
            cleanup_dataset(ds['local_dir'], name)
        continue

    # 3. Evaluate
    result = evaluator.evaluate_dataset(loader, name)
    all_results[name] = result
    print(f"  EER: {result['eer']:.2f}% [{result.get('ci_low', 0):.2f}–{result.get('ci_high', 0):.2f}%]")

    # 4. Cleanup to free disk (unless no_cleanup is set)
    del loader
    torch.cuda.empty_cache()
    if not ds.get('no_cleanup') and ds.get('local_dir'):
        cleanup_dataset(ds['local_dir'], name)

    # Show remaining disk
    import subprocess
    df_out = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
    print(f"  Disk: {df_out.stdout.strip().split(chr(10))[-1]}")

# --- Display results ---
print("\n" + "=" * 70)
print("EVALUATION RESULTS")
print("=" * 70)
md_table = MultiDatasetEvaluator.results_to_markdown(all_results)
print(md_table)
latex_table = MultiDatasetEvaluator.results_to_latex(all_results)
print("\n### LaTeX Table")
print(latex_table)

# --- Save to Drive ---
results_save_path = os.path.join(
    config.drive_log_dir,
    f'eval_{config.backbone}_results.json'
)
os.makedirs(os.path.dirname(results_save_path), exist_ok=True)
with open(results_save_path, 'w') as f:
    json.dump({
        'backbone': config.backbone,
        'results': all_results,
    }, f, indent=2, default=str)
print(f"\nResults saved to: {results_save_path}")


AMP dtype: torch.bfloat16
SEQUENTIAL EVALUATION (extract -> eval -> cleanup)

──────────────────────────────────────────────────
Dataset: ITW
──────────────────────────────────────────────────
  [ITW] Extracting from /content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar...
  [ITW] Done.
  In-the-Wild: 31779 samples (19963 bonafide, 11816 spoof)
  Loaded 31779 samples


Eval ITW:   0%|          | 0/663 [00:00<?, ?it/s]

  EER: 11.88% [11.55–12.27%]
  [ITW] Cleaned up /content/data/itw
  Disk: overlay         113G   63G   51G  56% /

──────────────────────────────────────────────────
Dataset: 2019_LA_eval
──────────────────────────────────────────────────
  [2019_LA_eval] Extracting from /content/drive/MyDrive/Jakob/Datasets/LA.tar...
  [2019_LA_eval] Done.
  ASVspoof 2019 LA Eval: 71237 samples (7355 bonafide, 63882 spoof)
  Loaded 71237 samples


Eval 2019_LA_eval:   0%|          | 0/1485 [00:00<?, ?it/s]

  EER: 1.66% [1.57–1.76%]
  Disk: overlay         113G   68G   46G  61% /

──────────────────────────────────────────────────
Dataset: 2019_LA_dev
──────────────────────────────────────────────────
  ASVspoof 2019 LA Eval: 24844 samples (2548 bonafide, 22296 spoof)
  Loaded 24844 samples


Eval 2019_LA_dev:   0%|          | 0/518 [00:00<?, ?it/s]

  EER: 0.24% [0.16–0.34%]
  [2019_LA_dev] Cleaned up /content/data/asvspoof
  Disk: overlay         113G   61G   53G  54% /

──────────────────────────────────────────────────
Dataset: 2021_LA
──────────────────────────────────────────────────
  [2021_LA] Already extracted at /content/data/asvspoof21_la
  Keys file: None
  Audio dir: /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac
  Extracted files (first 20):
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/LICENSE.txt
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/README.LA.txt
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/ASVspoof2021.LA.cm.eval.trl.txt
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac/LA_E_5035912.flac
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac/LA_E_9887153.flac
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac/LA_E_5696373.flac
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac/LA_E_3829282.flac
    /content/data/asvspoof21_la/ASVspoof2021_LA_eval/flac/L

## Multi-Model Comparison

Compare multiple trained checkpoints (different backbones, seeds, or ablation configurations) across all evaluation datasets. This produces the main results table for the thesis.

**Expected comparisons**:
- **Backbone comparison**: WavLM-Large vs XLSR-300M vs Data2Vec-Large vs Wav2Vec2-Robust
- **Seed stability**: Same backbone across seeds {42, 123, 2026} to assess variance
- **Ablation variants**: Full GLADA vs components removed (no GLADA, no VIB, no CKA, no speaker adversary)

### Why Multi-Model Comparison Matters

Single-model results can be misleading due to seed sensitivity (Rule 4 requires multi-seed experiments with statistical reporting). The comparison table shows:
1. Whether GLADA consistently improves over baselines across backbones
2. Whether the improvement is robust to random seed variation
3. Which components contribute most to performance (ablation analysis)

The results dictionary maps `model_name -> {dataset_name -> eval_metrics}` and is saved to JSON for later analysis.

In [ ]:
# ============================================================================
# Multi-Model Comparison: evaluate multiple checkpoints on all datasets
# Produces the main results table for the thesis (Table X in manuscript)
# ============================================================================

# --- CONFIGURE CHECKPOINTS HERE ---
# Map from display name -> (checkpoint_path, backbone_key)
# Update these paths to match your trained checkpoints.
CHECKPOINTS = {
    # Backbone comparison (same config, different SSL models)
    'WavLM-Large':      ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/20260407_032408/best_model.pt', 'wavlm-large'),
    # 'XLSR-300M':        ('/content/drive/MyDrive/Jakob/Experiments/glada_xlsr-300m_seed42/.../best_model.pt', 'xlsr-300m'),
    # 'Data2Vec-Large':   ('/content/drive/MyDrive/Jakob/Experiments/glada_data2vec-large_seed42/.../best_model.pt', 'data2vec-large'),
    # 'Wav2Vec2-Robust':  ('/content/drive/MyDrive/Jakob/Experiments/glada_wav2vec2-robust_seed42/.../best_model.pt', 'wav2vec2-robust'),

    # Seed stability (same backbone, different seeds)
    # 'WavLM-Large (s42)':   ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/.../best_model.pt', 'wavlm-large'),
    # 'WavLM-Large (s123)':  ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed123/.../best_model.pt', 'wavlm-large'),
    # 'WavLM-Large (s2026)': ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed2026/.../best_model.pt', 'wavlm-large'),

    # Ablation variants
    # 'Full GLADA':       ('/content/drive/MyDrive/Jakob/Experiments/glada_wavlm-large_seed42/.../best_model.pt', 'wavlm-large'),
    # 'No GLADA':         ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_glada/.../best_model.pt', 'wavlm-large'),
    # 'No VIB':           ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_vib/.../best_model.pt', 'wavlm-large'),
    # 'No CKA':           ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_cka/.../best_model.pt', 'wavlm-large'),
    # 'No Speaker GRL':   ('/content/drive/MyDrive/Jakob/Experiments/ablation_no_speaker/.../best_model.pt', 'wavlm-large'),
}

if len(CHECKPOINTS) == 0:
    print("No checkpoints configured for multi-model comparison.")
    print("Uncomment and update the checkpoint paths above, then re-run this cell.")
else:
    comparison_results = {}  # {model_name: {dataset_name: result_dict}}

    for model_name, (ckpt_path, backbone_key) in CHECKPOINTS.items():
        print(f"\n{'='*70}")
        print(f"Evaluating: {model_name}")
        print(f"{'='*70}")

        try:
            # Load model
            m, cfg = load_model(ckpt_path, backbone_key)
            m = m.to(device)

            # Evaluate on all datasets
            ev = MultiDatasetEvaluator(m, device, amp_dtype=amp_dtype)
            comparison_results[model_name] = ev.evaluate_all(eval_datasets)

            # Free GPU memory for next model
            del m, ev
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"  ERROR loading {model_name}: {e}")
            comparison_results[model_name] = {'error': str(e)}

    # --- Print comparison table ---
    print("\n" + "="*70)
    print("MULTI-MODEL COMPARISON")
    print("="*70)

    # Collect all dataset names across all models
    all_dataset_names = set()
    for model_name, res in comparison_results.items():
        if 'error' not in res:
            all_dataset_names.update(res.keys())
    all_dataset_names = sorted(all_dataset_names)

    # Markdown comparison table
    header = "| Model | " + " | ".join([f"{d} EER (%)" for d in all_dataset_names]) + " |"
    separator = "|" + "|".join(["---"] * (1 + len(all_dataset_names))) + "|"
    print(header)
    print(separator)

    for model_name, res in comparison_results.items():
        if 'error' in res:
            row = f"| {model_name} | " + " | ".join(["ERROR"] * len(all_dataset_names)) + " |"
        else:
            cells = []
            for d in all_dataset_names:
                if d in res:
                    r = res[d]
                    cells.append(f"{r['eer']:.2f} [{r.get('ci_low', r['eer']):.2f}, {r.get('ci_high', r['eer']):.2f}]")
                else:
                    cells.append("N/A")
            row = f"| {model_name} | " + " | ".join(cells) + " |"
        print(row)

    # --- LaTeX comparison table ---
    print("\n### LaTeX Comparison Table")
    n_cols = 1 + len(all_dataset_names)
    col_spec = "l" + "c" * len(all_dataset_names)
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Multi-model comparison. EER (\%) with 95\% bootstrap CI.}")
    print(r"\label{tab:multi_model}")
    print(f"\\begin{{tabular}}{{{col_spec}}}")
    print(r"\toprule")
    print("Model & " + " & ".join(all_dataset_names) + r" \\")
    print(r"\midrule")
    for model_name, res in comparison_results.items():
        if 'error' in res:
            row_cells = ["--"] * len(all_dataset_names)
        else:
            row_cells = []
            for d in all_dataset_names:
                if d in res:
                    r = res[d]
                    row_cells.append(f"{r['eer']:.2f}")
                else:
                    row_cells.append("--")
        print(f"{model_name} & " + " & ".join(row_cells) + r" \\")
    print(r"\bottomrule")
    print(r"\end{tabular}")
    print(r"\end{table}")

    # --- Save comparison results to JSON ---
    comparison_save_path = os.path.join(
        config.drive_log_dir, 'multi_model_comparison.json'
    )
    with open(comparison_save_path, 'w') as f:
        json.dump(comparison_results, f, indent=2, default=str)
    print(f"\nComparison results saved to: {comparison_save_path}")

## SOTA Comparison

Published results from the literature for comparison. Values are EER (%) on standard evaluation sets.

**Sources**: IB-CAAN (Huang et al., 2025), HierCon (Liang et al., 2026), MultiConv+CKA (Tran et al., 2025), AASIST (Jung et al., 2022), RawNet2 (Tak et al., 2021).

### Notes on Comparison Fairness

- All systems listed train on ASVspoof 2019 LA and evaluate on ITW (Muller et al., 2022)
- Some systems (IB-CAAN, MultiConv) use additional augmentation or ensembles
- Our results use the same evaluation protocol for direct comparison
- AASIST and RawNet2 use fixed-architecture backends (no SSL backbone)
- SSL-based methods (HierCon, IB-CAAN, GLADA) generally outperform fixed-architecture methods on cross-domain benchmarks

In [ ]:
# ============================================================================
# SOTA Comparison Table
# Published EER (%) results from the literature vs our GLADA results.
# All methods train on ASVspoof 2019 LA; ITW is the cross-domain benchmark.
# ============================================================================

# --- Published SOTA results ---
# Format: (method, backbone, ITW EER %, ASVspoof 2019 LA EER %, reference)
SOTA_RESULTS = [
    # Method              Backbone             ITW EER   LA EER   Reference
    ('RawNet2',          'RawNet2 (fixed)',     31.20,    5.13,   'Tak et al., Interspeech 2021'),
    ('AASIST',           'AASIST (fixed)',      24.70,    0.83,   'Jung et al., ICASSP 2022'),
    ('AASIST + SSL',     'Wav2Vec2-XLSR',      10.80,    0.90,   'Tak et al., Interspeech 2022'),
    ('HierCon',          'XLS-R-300M',           6.87,    None,   'Liang et al., arXiv 2602.01032, 2026'),
    ('MultiConv+CKA',    'WavLM-Large',          4.44,    None,   'Tran et al., ACM MM 2025'),
    ('IB-CAAN',          'WavLM-Large',          4.93,    None,   'Huang et al., arXiv 2509.23618, 2025'),
]

# --- Our GLADA results (populated from evaluation above) ---
# If multi-model comparison was run, use those results;
# otherwise fall back to the single-model evaluation.
our_results = []

if 'comparison_results' in dir() and len(comparison_results) > 0:
    # Use multi-model comparison results
    for model_name, res in comparison_results.items():
        if 'error' in res:
            continue
        itw_eer = res.get('ITW', {}).get('eer', None)
        la_eer = res.get('2019_LA_eval', {}).get('eer', None)
        our_results.append((
            f'GLADA ({model_name})',
            model_name,
            itw_eer,
            la_eer,
            'Ours'
        ))
elif 'all_results' in dir() and len(all_results) > 0:
    # Use single-model evaluation results
    itw_eer = all_results.get('ITW', {}).get('eer', None)
    la_eer = all_results.get('2019_LA_eval', {}).get('eer', None)
    our_results.append((
        f'GLADA (ours)',
        config.backbone,
        itw_eer,
        la_eer,
        'Ours'
    ))

# --- Markdown SOTA table ---
print("### SOTA Comparison (Markdown)")
print("| Method | Backbone | ITW EER (%) | LA EER (%) | Reference |")
print("|--------|----------|-------------|------------|-----------|")
for method, backbone, itw, la, ref in SOTA_RESULTS:
    itw_str = f"{itw:.2f}" if itw is not None else "--"
    la_str = f"{la:.2f}" if la is not None else "--"
    print(f"| {method} | {backbone} | {itw_str} | {la_str} | {ref} |")
# Our results (highlighted)
for method, backbone, itw, la, ref in our_results:
    itw_str = f"**{itw:.2f}**" if itw is not None else "--"
    la_str = f"**{la:.2f}**" if la is not None else "--"
    print(f"| {method} | {backbone} | {itw_str} | {la_str} | {ref} |")

# --- LaTeX SOTA table ---
print("\n### SOTA Comparison (LaTeX -- copy to thesis)")
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{Comparison with published methods. EER (\%) on ASVspoof 2019 LA eval and In-the-Wild (ITW) datasets. All methods train on ASVspoof 2019 LA. Lower is better.}")
print(r"\label{tab:sota_comparison}")
print(r"\begin{tabular}{llcc}")
print(r"\toprule")
print(r"Method & Backbone & ITW EER (\%) & LA EER (\%) \\")
print(r"\midrule")
for method, backbone, itw, la, ref in SOTA_RESULTS:
    itw_str = f"{itw:.2f}" if itw is not None else "--"
    la_str = f"{la:.2f}" if la is not None else "--"
    print(f"{method}~\\cite{{{ref.split(',')[0].lower().replace(' ', '_')}}} & {backbone} & {itw_str} & {la_str} \\\\")
print(r"\midrule")
for method, backbone, itw, la, ref in our_results:
    itw_str = f"\\textbf{{{itw:.2f}}}" if itw is not None else "--"
    la_str = f"\\textbf{{{la:.2f}}}" if la is not None else "--"
    print(f"{method} & {backbone} & {itw_str} & {la_str} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# --- Save complete SOTA comparison to Drive ---
sota_save_path = os.path.join(config.drive_log_dir, 'sota_comparison.json')
os.makedirs(os.path.dirname(sota_save_path), exist_ok=True)
sota_data = {
    'published': [
        {'method': m, 'backbone': b, 'itw_eer': i, 'la_eer': l, 'reference': r}
        for m, b, i, l, r in SOTA_RESULTS
    ],
    'ours': [
        {'method': m, 'backbone': b, 'itw_eer': i, 'la_eer': l, 'reference': r}
        for m, b, i, l, r in our_results
    ],
}
with open(sota_save_path, 'w') as f:
    json.dump(sota_data, f, indent=2, default=str)
print(f"\nSOTA comparison saved to: {sota_save_path}")